In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from rich import print

# ============================================================
# SAMPLE PARAGRAPHS
# ============================================================

paragraphs = [
    "PostgreSQL supports streaming replication.",
    "Replication improves database reliability.",
    "Database backups help disaster recovery.",

    "Transformers generate semantic embeddings.",
    "Vector databases enable semantic search.",
    "Embeddings capture contextual meaning.",

    "Enterprise pricing starts at $200/month.",
    "Dedicated support includes onboarding.",
]

# ============================================================
# LOAD EMBEDDING MODEL
# ============================================================

model = SentenceTransformer("BAAI/bge-small-en")

# ============================================================
# DETECT TOPIC SHIFTS
# ============================================================

def detect_topic_shifts(paragraphs, model, threshold=0.7):

    # Generate embeddings
    embeddings = model.encode(paragraphs)

    splits = []

    print("\n[bold green]Similarity Scores[/bold green]\n")

    # Compare adjacent paragraphs
    for i in range(len(paragraphs) - 1):

        sim = cosine_similarity(
            [embeddings[i]],
            [embeddings[i + 1]]
        )[0][0]

        print(
            f"[cyan]{i} -> {i+1}[/cyan] "
            f"Similarity = [yellow]{sim:.4f}[/yellow]"
        )

        # Detect topic shift
        if sim < threshold:

            print("[red]TOPIC SHIFT DETECTED[/red]\n")

            splits.append(i)

    return splits

# ============================================================
# RUN
# ============================================================

splits = detect_topic_shifts(
    paragraphs=paragraphs,
    model=model,
    threshold=0.75,
)

# ============================================================
# DISPLAY SPLITS
# ============================================================

print("\n[bold blue]Topic Shift Positions[/bold blue]")
print(splits)

# ============================================================
# BUILD CHUNKS
# ============================================================

chunks = []

start = 0

for split_idx in splits:

    chunk = paragraphs[start:split_idx + 1]
    chunks.append(chunk)

    start = split_idx + 1

# Add final chunk
chunks.append(paragraphs[start:])

# ============================================================
# DISPLAY CHUNKS
# ============================================================

print("\n[bold green]Semantic Chunks[/bold green]\n")

for idx, chunk in enumerate(chunks):

    print(f"[bold yellow]Chunk {idx+1}[/bold yellow]")

    for sentence in chunk:
        print(f"• {sentence}")

    print()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6056.91it/s]


Similarity Scores

0 -> 1 Similarity = 0.9166

1 -> 2 Similarity = 0.8902

2 -> 3 Similarity = 0.7856

...

• Vector databases enable semantic search.

• Embeddings capture contextual meaning.

Chunk 2

• Enterprise pricing starts at $200/month.

• Dedicated support includes onboarding.